# Nutosis

It is October 2018. The squirrels in Central Park are running into a problem and we need your help.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help).

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes the train or test set and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preparation should be noted here but actually implemented in the Preparation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit the train/test parquet files and a pickled preprocessor and final model along with these notebooks.

# Frame the Problem and Look at the Big Picture

1. **Define the objective in business terms.** 
The objective is to predict if a squirrel has nutosis based on symptoms provided in order to stop the spread of the lethal disease.
2. **How will your solution be used?** 
The model's predictions will be used to inform the department where workers need to be dispatched. They will be dispatched in areas where a squirrel assumed to have nutosis was spotted. They will capture the squire and administer a treatment in order to cure the squirrel of nutosis and stop the spread.
3. **What are the current solutions/workarounds (if any)?** 
Currently there is a mobile app anyone can download and use to report squirrel sightings. These sightings contain a section where users can note what they noticed about the squirrel as a text entry. Workers can look at these text entries and decide if a squirrel is likely to have nutosis and go capture it based on its location in the sighting post.
4. **How should you frame this problem (supervised/unsupervised, online/offline, …)?** 
Supervised, online, classification. Supervised because there are labels in the data we currently have (whether or not a squirrel is healthy). Online because we will need to add data to this consistently. Classification because the prediction being made is if the squirrel is percieved to have nutosis or not.
5. **How should performance be measured? Is the performance measure aligned with the business objective?** 
Performance should be measured using a combination of recall and precision. We should favor recall over precision because its safer to overpredict than to miss out on a potential squirrel with nutosis. This is because treatment is not that expensive and the disease is quite infectious.
6. **What would be the minimum performance needed to reach the business objective?** 
As close to 100 percent recall as possible while maintaining a healthy budget for treatments. So recall is  more important than precision, but not by a lot.
7. **What are comparable problems? Can you reuse experience or tools?** 
Some comparable problems are the bike sharing problem and the previous early spring project. These two problems are classification problems however the bike sharing was 3 classes, and not binary like this and the early spring project are. We can use some of the techniques for getting and exploring the data that we used on the early spring data as it is also dealing with similar binary classification.
8. **Is human expertise available?** 
Yes, human expertise is available, we have a member of the group that is working to stop the spread of nutosis. They have information regarding the symptoms, and data. So they are the point of contact for all domain related questions. We also have Professor Bush to assist with Machine Learning related questions. 
9. **How would you solve the problem manually?** 
Like mentioned before, there is a mobile app containing squirrel sighting data. The idea would be to look at what the symptoms are of nutosis, and if a users sighting description shows enough signs that match with the symptoms, send a worker out to get the squirrel and take them back to administer the treatment.
10. **List the assumptions you (or others) have made so far. Verify assumptions if possible.** 
We are assuming that it is okay to mislabel a squirrel with nutosis as long as some symptoms are present. Its assumed to be safe than sorry.

# Get the Data

Our two extra sources of data are OpenMeteo from the same timeframe for weather features, and the hectare data from the same squirrel central park census. 

We chose OpenMeteo for weather because we think certain squirrel behavior, whether it is usual or unusual might be explainable by the weather and not because they are diseased. For example: say it is a low temperature, a squirrel being out is unusual and should line up with squirrels who have the neurological disease.

The hectare data was chosen because it could explain certain unusual squirrel behavior with where the sighting is specifically in central park and what other associations there may be with the location. Things like litter, other animal sightings and density of people in the area. For example, litter could be an alterier reasoning behind why squirrels might be friendly and more eager to approach humans.

**NOTE: we might want to process text for the freetext entries within the other squirrel census data or drop the columns.**

In [2]:
from pathlib import Path
import requests
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [3]:
def fetch_and_cache_nyc_csv(
    dataset_id: str,
    filename: str,
    cache_dir: str = "data_cache",
) -> Path:
    """
    Fetch an entire NYC Open Data table (bypassing the 1,000-row API cap) via its
    bulk CSV export URL and cache it locally.
    """
    cache_path = Path(cache_dir) / filename
    if not cache_path.exists():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://data.cityofnewyork.us/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD"
        df = pd.read_csv(url)
        df.to_csv(cache_path, index=False)
        print(f"Fetched and cached {cache_path} ({len(df)} rows)")
    else:
        print(f"Loaded cached {cache_path}")

    return cache_path


def fetch_and_cache_weather_data(cache_dir: str = "data_cache") -> Path:
    """
    Fetch Central Park hourly historical weather (Oct 6–20, 2018) from Open-Meteo
    (no API key required), aggregate into AM/PM shifts, and cache locally.
    """
    cache_path = Path(cache_dir) / "central_park_weather_oct2018.csv"
    if not cache_path.exists():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        url = (
            "https://archive-api.open-meteo.com/v1/archive"
            "?latitude=40.7812&longitude=-73.9665"
            "&start_date=2018-10-06&end_date=2018-10-20"
            "&hourly=temperature_2m,precipitation,wind_speed_10m"
            "&timezone=America%2FNew_York"
        )
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        hourly = pd.DataFrame(resp.json()["hourly"])

        hourly["dt"] = pd.to_datetime(hourly["time"])
        hourly["Date"] = hourly["dt"].dt.strftime("%Y-%m-%d")
        hourly["hour"] = hourly["dt"].dt.hour

        # Map hours to census shifts: AM (7am–12pm) and PM (1pm–6pm)
        hourly["Shift"] = None
        hourly.loc[hourly["hour"].between(7, 12), "Shift"] = "AM"
        hourly.loc[hourly["hour"].between(13, 18), "Shift"] = "PM"

        shift_weather = (
            hourly.dropna(subset=["Shift"])
            .groupby(["Date", "Shift"], as_index=False)
            .agg(
                shift_temp_c=("temperature_2m", "mean"),
                shift_precip_mm=("precipitation", "sum"),
                shift_wind_kmh=("wind_speed_10m", "mean"),
            )
        )
        shift_weather.to_csv(cache_path, index=False)
        print(f"Fetched and cached {cache_path} ({len(shift_weather)} rows)")
    else:
        print(f"Loaded cached {cache_path}")

    return cache_path


def _parse_census_date(series: pd.Series) -> pd.Series:
    """Normalize MMDDYYYY integers/strings (e.g., 10142018) or ISO dates to datetime64."""
    s = series.astype(str).str.strip()
    is_mmddyyyy = s.str.fullmatch(r"\d{7,8}")
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    if is_mmddyyyy.any():
        parsed.loc[is_mmddyyyy] = pd.to_datetime(
            s.loc[is_mmddyyyy].str.zfill(8), format="%m%d%Y", errors="coerce"
        )
    if (~is_mmddyyyy).any():
        parsed.loc[~is_mmddyyyy] = pd.to_datetime(s.loc[~is_mmddyyyy], errors="coerce")

    return parsed.dt.normalize()

def load_and_merge_datasets(
    squirrel_csv_path: str | Path,
    diseased_csv_path: str | Path,
    context_csv_paths: str | Path | list[str | Path] | None = None,
) -> pd.DataFrame:
    """
    Merge squirrel sightings, headerless diseased IDs, and additional context CSVs
    (Hectare Data and Shift Weather Data) while preserving exact row count.
    """
    # 1. Load primary squirrel sightings table
    df = pd.read_csv(squirrel_csv_path)
    df["Unique Squirrel ID"] = df["Unique Squirrel ID"].astype(str).str.strip()
    df["Date_parsed"] = _parse_census_date(df["Date"])
    df["Shift"] = df["Shift"].astype(str).str.strip().str.upper()
    df["Hectare"] = df["Hectare"].astype(str).str.strip().str.upper()

    # 2. Load diseased IDs (header=None preserves row 1: '11H-AM-1010-03')
    df_dis = pd.read_csv(diseased_csv_path, header=None, names=["Unique Squirrel ID"])
    diseased_ids = set(df_dis["Unique Squirrel ID"].astype(str).str.strip())
    df["is_diseased"] = df["Unique Squirrel ID"].isin(diseased_ids).astype(int)

    # 3. Left-join additional context datasets
    if context_csv_paths is not None:
        if isinstance(context_csv_paths, (str, Path)):
            context_csv_paths = [context_csv_paths]

        for i, path in enumerate(context_csv_paths, start=1):
            df_ctx = pd.read_csv(path)
            df_ctx["Date_parsed"] = _parse_census_date(df_ctx["Date"])

            if "Shift" in df_ctx.columns:
                df_ctx["Shift"] = df_ctx["Shift"].astype(str).str.strip().str.upper()

            if "Hectare" in df_ctx.columns:
                df_ctx["Hectare"] = df_ctx["Hectare"].astype(str).str.strip().str.upper()
                join_keys = ["Hectare", "Date_parsed", "Shift"]
            elif "Shift" in df_ctx.columns:
                join_keys = ["Date_parsed", "Shift"]
            else:
                join_keys = ["Date_parsed"]

            overlap_cols = [c for c in ["Date", "Hectare Squirrel Number"] if c in df_ctx.columns]
            df_ctx = df_ctx.drop(columns=overlap_cols).drop_duplicates(subset=join_keys, keep="first")

            n_before = len(df)
            df = df.merge(df_ctx, on=join_keys, how="left", suffixes=("", f"_ctx{i}"))
            assert len(df) == n_before, f"Merge altered row count: {n_before} -> {len(df)}"

    # 4. Cast mixed object columns to nullable string dtype for clean Parquet serialization
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string")

    return df

def split_and_save_parquet(
    df: pd.DataFrame,
    target_col: str = "is_diseased",
    group_col: str = "Unique Squirrel ID",
    n_splits: int = 5,  # 5 splits = 80% train / 20% test
    random_state: int = 42,
    train_path: str = "train_set.parquet",
    test_path: str = "test_set.parquet",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create a stratified train/test split grouped by Unique Squirrel ID (preventing
    duplicate sightings of the same squirrel from leaking across splits) and save to Parquet.
    """
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    train_idx, test_idx = next(sgkf.split(df, df[target_col], groups=df[group_col]))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    train_df.to_parquet(train_path, index=False)
    test_df.to_parquet(test_path, index=False)

    print(f"Saved {train_path}: {train_df.shape} | Positive rate: {train_df[target_col].mean():.2%}")
    print(f"Saved {test_path}:  {test_df.shape} | Positive rate: {test_df[target_col].mean():.2%}")
    return train_df, test_df

In [4]:
squirrel_csv = fetch_and_cache_nyc_csv("vfnx-vebw", "squirrel_data_2018.csv")
hectare_csv = fetch_and_cache_nyc_csv("ej9h-v6g2", "hectare_data_2018.csv")
weather_csv = fetch_and_cache_weather_data()

df_merged = load_and_merge_datasets(
    squirrel_csv_path=squirrel_csv,
    diseased_csv_path="diseased.csv",
    context_csv_paths=[hectare_csv, weather_csv],
)

train_df, test_df = split_and_save_parquet(
    df_merged,
    train_path="train_set.parquet",
    test_path="test_set.parquet",
)

Fetched and cached data_cache/squirrel_data_2018.csv (3023 rows)
Fetched and cached data_cache/hectare_data_2018.csv (700 rows)
Fetched and cached data_cache/central_park_weather_oct2018.csv (30 rows)


/var/folders/mw/b7c8xl6s051dz_4qjhppf1l00000gn/T/ipykernel_85320/2913195010.py:132: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Saved train_set.parquet: (2418, 46) | Positive rate: 12.53%
Saved test_set.parquet:  (605, 46) | Positive rate: 12.56%


We have stratified based on the target label: is_diseased.